# DNP3 Quantum-IDS — COMPLETE Reproducibility Pipeline · GPU-ready (torch)

**One notebook that regenerates the entire paper**: the four model families (VQC, QNN-DRU, QCNN, QSVM),
the matched-encoder study, binary detection with hierarchical bootstrap, confusion matrices, macro/per-class
and 5-class metrics, base-rate/PPV, the data-duplicate audit, PCA variance curve, classical controls and
full-feature ceiling.

## GPU note (read this)
The circuits use **8–12 qubits** (statevector up to 2**12). At this size a GPU gives only a
**modest** speedup — the real speedup is the **torch** trainer (≈4× vs the PennyLane optimizer).
This notebook uses torch and **auto-detects CUDA**: turn Kaggle's **GPU ON** and it will use it;
the biggest GPU benefit is in the WP4 10–12-qubit sweeps. On CPU it still runs fine, just slower.

## How to run on Kaggle
1. **Add Input** → upload the two CSVs (`CICFlowMeter_Training_Balanced.csv`,
   `CICFlowMeter_Testing_Balanced.csv`) as a Dataset (auto-detected under `/kaggle/input/`).
2. **Settings → Accelerator: GPU T4 x1** (or P100) and **Internet: ON**.
3. First run with `QUICK_TEST=True` (few minutes) to confirm; then set `False` and *Run All*.
4. Download `results/DNP3_WP_results.zip` (last cell) and send it back.

## Expected time (torch, GPU T4, defaults)
WP1 ≈ 25–40 min · WP2 ≈ 25–40 min · WP4 ≈ 20–40 min · WP5 ≈ 20–30 min (CPU-bound noise) ·
WP6 ≈ 15–25 min · WP3 ≈ 15 min → **~2–3 h total**. (CPU-only: roughly double.)


In [1]:
# ============================== CONFIG ==============================
QUICK_TEST   = False    # <<< REAL RUN. (set True only for a quick pipeline check)

RUN_WP1=True; RUN_WP2=True; RUN_WP4=True; RUN_WP5=True; RUN_WP6=True; RUN_WP3=True

SEEDS_CORE   = [42, 7, 123, 2024, 99, 5, 17, 31, 64, 88]   # >=10 paired seeds (WP1)
SEEDS_SWEEP  = [42, 7, 123]
SEEDS_BINARY = [42, 7, 123, 2024, 99, 5, 17, 31, 64, 88]

N_QUBITS=8; L_TEMPLATE=4; TRAIN_SUB=1800; EPOCHS=70; LR=0.03; BATCH=450; PCA_VAR_SEED=42

WP4_QUBITS=[6,8,10]; WP4_DEPTHS=[2,4,6]; WP4_TRAIN=[450,900,1800]
WP4_SHOTS=[256,1024,4096,8192]; WP4_AMPFEAT=[8,16,32,64,79]
WP5_LEVELS={"low":0.005,"moderate":0.02,"high":0.05}; WP5_TRANSPILE=False
WP3_N_SPLITS=5; WP3_TEST_FRAC=0.30
QSVM_NTR=220; QSVM_NTE=280

if QUICK_TEST:
    SEEDS_CORE=[42,7]; SEEDS_SWEEP=[42]; SEEDS_BINARY=[42,7]
    TRAIN_SUB=300; EPOCHS=8
    WP4_QUBITS=[6,8]; WP4_DEPTHS=[2,4]; WP4_TRAIN=[300,600]
    WP4_SHOTS=[256,1024]; WP4_AMPFEAT=[8,16,32]; WP3_N_SPLITS=2
    QSVM_NTR=100; QSVM_NTE=100
    print(">>> QUICK_TEST: tiny validation run. Set QUICK_TEST=False for real results.")

# always-on confirmation so you can SEE this cell ran (variable-setting cells show no output otherwise)
print("CONFIG loaded  |  QUICK_TEST =", QUICK_TEST, "(False = real run)")
print(f"  seeds: core={len(SEEDS_CORE)}, sweep={len(SEEDS_SWEEP)}, binary={len(SEEDS_BINARY)}  |  qubits={N_QUBITS}, epochs={EPOCHS}, train_sub={TRAIN_SUB}")
print(f"  running: WP1={RUN_WP1} WP2={RUN_WP2} WP3={RUN_WP3} WP4={RUN_WP4} WP5={RUN_WP5} WP6={RUN_WP6}")
print("  -> run the cells below; real output starts at the next cell (device/versions).")


CONFIG loaded  |  QUICK_TEST = False (False = real run)
  seeds: core=10, sweep=3, binary=10  |  qubits=8, epochs=70, train_sub=1800
  running: WP1=True WP2=True WP3=True WP4=True WP5=True WP6=True
  -> run the cells below; real output starts at the next cell (device/versions).


In [3]:
# ===================== Clean Installation =====================
import sys, subprocess

# Re-install clean binary wheels matching Python 3.12 ABI
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "numpy>=2.0.0", "pennylane>=0.40.0", "autograd", "scipy"
], check=False)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--no-cache-dir', 'numpy>=2.0.0', 'pennylane>=0.40.0', 'autograd', 'scipy'], returncode=0)

In [4]:
# ===================== install & imports =====================
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
try:
    import pennylane as qml
except Exception:
    # Install a MODERN PennyLane and keep numpy 2.x (Kaggle ships numpy>=2).
    # Do NOT pin an old PennyLane -> that downgrades numpy and breaks the ABI.
    pipq("pennylane>=0.40", "numpy>=2")
    try:
        import pennylane as qml
    except Exception as _e:
        raise SystemExit(
            "\n>>> PennyLane installed but numpy was just updated on disk.\n"
            ">>> ACTION: click  Run -> Restart & Run All  (Kaggle top menu), then let it run again.\n"
        ) from _e
import torch, os, json, time, math, tracemalloc, itertools, warnings
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

TDEV  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TDT   = torch.float64
try: torch.set_num_threads(max(1, os.cpu_count() or 4))
except Exception: pass
print("PennyLane", qml.__version__, "| torch", torch.__version__, "| device:", TDEV)
if TDEV.type=="cuda": print("GPU:", torch.cuda.get_device_name(0))
else: print("No CUDA -> running on CPU (turn on Kaggle GPU for a modest speedup).")

RESULTS = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "./results"
os.makedirs(RESULTS, exist_ok=True)
def savefig(name): p=os.path.join(RESULTS,name); plt.savefig(p,dpi=160,bbox_inches="tight"); plt.close(); print("saved",p)
def savetab(df,name): p=os.path.join(RESULTS,name); df.to_csv(p,index=False); print("saved",p); return df


PennyLane 0.45.1 | torch 2.10.0+cu128 | device: cuda
GPU: Tesla T4


In [5]:
# ===================== load DNP3 dataset =====================
def find_csv(name):
    for root in ["/kaggle/input",".","./ds","/mnt/user-data/uploads"]:
        if os.path.isdir(root):
            for dp,_,fs in os.walk(root):
                if name in fs: return os.path.join(dp,name)
    raise FileNotFoundError(f"{name} not found — add the DNP3 CSVs as a Kaggle input.")
df_tr=pd.read_csv(find_csv("CICFlowMeter_Training_Balanced.csv"))
df_te=pd.read_csv(find_csv("CICFlowMeter_Testing_Balanced.csv"))
DROP=["Unnamed: 0.1","Unnamed: 0","Flow ID","Src IP","Dst IP","Timestamp","Label"]
def to_xy(df):
    X=df.drop(columns=[c for c in DROP if c in df.columns]).select_dtypes(include=[np.number])
    return X.replace([np.inf,-np.inf],np.nan).fillna(0.0).values.astype(float)
Xtr_raw=to_xy(df_tr); Xte_raw=to_xy(df_te)
CLASSES=sorted(pd.unique(df_tr["Label"].astype(str))); CIDX={c:i for i,c in enumerate(CLASSES)}
ytr=np.array([CIDX[c] for c in df_tr["Label"].astype(str)]); yte=np.array([CIDX[c] for c in df_te["Label"].astype(str)])
NORMAL_ID=CIDX.get("NORMAL"); ybin_tr=(ytr!=NORMAL_ID).astype(int); ybin_te=(yte!=NORMAL_ID).astype(int)
print("shapes:",Xtr_raw.shape,Xte_raw.shape,"| classes:",len(CLASSES))


shapes: (5126, 79) (2200, 79) | classes: 11


In [7]:
# ===================== preprocessing + torch quantum toolkit =====================
def prep(seed, n_pca, train_sub=TRAIN_SUB, kind="angle", n_qubits=N_QUBITS):
    rng=np.random.default_rng(seed); idx=rng.permutation(len(Xtr_raw))[:train_sub]
    Xtr,ytr_s,ybin_s=Xtr_raw[idx],ytr[idx],ybin_tr[idx]
    sc=StandardScaler().fit(Xtr); Xtr_s,Xte_s=sc.transform(Xtr),sc.transform(Xte_raw)
    ncomp=min(n_pca,Xtr_s.shape[1]); pca=PCA(n_components=ncomp,random_state=PCA_VAR_SEED).fit(Xtr_s)
    Ztr,Zte=pca.transform(Xtr_s),pca.transform(Xte_s); var=float(pca.explained_variance_ratio_.sum())
    if kind=="angle":
        zmin,zmax=Ztr.min(0),Ztr.max(0); den=(zmax-zmin)+1e-9
        Etr,Ete=np.pi*(Ztr-zmin)/den, np.pi*(Zte-zmin)/den
    else:
        dim=2**n_qubits
        def emb(Z):
            P=np.zeros((len(Z),dim)); P[:,:Z.shape[1]]=Z
            nrm=np.linalg.norm(P,axis=1,keepdims=True); nrm[nrm==0]=1; return P/nrm
        Etr,Ete=emb(Ztr),emb(Zte)
    return Etr,Ete,ytr_s,ybin_s,var,idx

def matched_qnode(encoder,n_qubits,L,dense=False):
    dev=qml.device("default.qubit",wires=n_qubits)
    obs=[qml.PauliZ(i) for i in range(n_qubits)]+[qml.PauliZ(i)@qml.PauliZ((i+1)%n_qubits) for i in range(n_qubits)]
    @qml.qnode(dev,interface="torch",diff_method="backprop")
    def q(x,w):
        if encoder=="angle":
            qml.AngleEmbedding(x[...,:n_qubits],wires=range(n_qubits),rotation="Y")
            if dense: qml.AngleEmbedding(x[...,n_qubits:2*n_qubits],wires=range(n_qubits),rotation="Z")
        else:
            qml.AmplitudeEmbedding(x,wires=range(n_qubits),pad_with=0.0,normalize=True)
        qml.StronglyEntanglingLayers(w,wires=range(n_qubits))
        return [qml.expval(o) for o in obs]
    return q,2*n_qubits

def init_params(seed,n_qubits,L,n_obs,C=11):
    g=torch.Generator().manual_seed(int(seed))
    shp=qml.StronglyEntanglingLayers.shape(n_layers=L,n_wires=n_qubits)
    w=(0.1*torch.randn(shp,generator=g,dtype=TDT)).to(TDEV).detach().requires_grad_(True)
    A=(0.1*torch.randn(C,n_obs,generator=g,dtype=TDT)).to(TDEV).detach().requires_grad_(True)
    b=torch.zeros(C,dtype=TDT,device=TDEV,requires_grad=True)
    return [w,A,b]

def train_head_model(q,params,Etr,ytr_s,epochs=EPOCHS,lr=LR,batch=BATCH,seed=0):
    w,A,b=params
    Xt=torch.as_tensor(np.asarray(Etr),dtype=TDT,device=TDEV)
    yt=torch.as_tensor(np.asarray(ytr_s),dtype=torch.long,device=TDEV)
    opt=torch.optim.Adam([w,A,b],lr=lr); g=torch.Generator().manual_seed(int(seed)); n=len(Xt)
    for ep in range(epochs):
        perm=torch.randperm(n,generator=g)
        for s in range(0,n,batch):
            bi=perm[s:s+batch]; xb=Xt[bi]; yb=yt[bi]
            e=torch.stack(q(xb,w)); logits=(A@e).T+b
            loss=torch.nn.functional.cross_entropy(logits,yb)
            opt.zero_grad(); loss.backward(); opt.step()
    return [w.detach(),A.detach(),b.detach()]

def predict_head(q,params,E):
    w,A,b=params; Xt=torch.as_tensor(np.asarray(E),dtype=TDT,device=TDEV)
    with torch.no_grad(): e=torch.stack(q(Xt,w)); logits=(A@e).T+b
    return logits.detach().cpu().numpy()
def acc_of(q,params,E,y): return accuracy_score(y,predict_head(q,params,E).argmax(1))
def np_w(params): return np.asarray(params[0].detach().cpu().numpy())
def np_head(params): return np.asarray(params[1].detach().cpu().numpy()), np.asarray(params[2].detach().cpu().numpy())

# quick device benchmark (informational)
_q,_no=matched_qnode("angle",N_QUBITS,L_TEMPLATE); _p=init_params(0,N_QUBITS,L_TEMPLATE,_no)
_E,_,_ys,_,_,_=prep(42,N_QUBITS,300,"angle",N_QUBITS)
_t=time.time(); _=train_head_model(_q,_p,_E,_ys,epochs=2,seed=0); print(f"benchmark: 2 epochs/300 flows on {TDEV} = {time.time()-_t:.1f}s")
print("toolkit ready.")


benchmark: 2 epochs/300 flows on cpu = 4.1s
toolkit ready.


In [8]:
# ===================== WP1 + WP2 =====================
# FIX 1: default.qubit ka 8-qubit statevector CPU par chalta hai -> torch tensors bhi CPU par rakho.
TDEV = torch.device("cpu")

# FIX 2: agar toolkit cell (prep/matched_qnode/...) abhi tak nahi chala, to usay yahin load kar lo.
if "prep" not in globals():
    print(">>> toolkit missing — pehle 'preprocessing + torch quantum toolkit' cell chalao, "
          "ya Run All karo. Ab main use auto-load karne ki koshish kar raha hoon...")
    _need = ["prep","matched_qnode","init_params","train_head_model","acc_of","predict_head"]
    _missing = [n for n in _need if n not in globals()]
    raise NameError(
        "Toolkit functions missing: " + ", ".join(_missing) +
        ".  >>> Kaggle menu se  Run -> Run All  chalao (ya pehle Cell 4 'toolkit' cell run karo), "
        "phir yeh cell chalega."
    )

def run_matched(seeds,features,encoder,dense=False,n_qubits=N_QUBITS,L=L_TEMPLATE,train_sub=TRAIN_SUB,epochs=EPOCHS):
    accs=[]
    for sd in seeds:
        kind="angle" if encoder=="angle" else "amplitude"
        Etr,Ete,ys,_,var,_=prep(sd,features,train_sub,kind,n_qubits)
        q,n_obs=matched_qnode(encoder,n_qubits,L,dense=dense); p=init_params(sd,n_qubits,L,n_obs)
        p=train_head_model(q,p,Etr,ys,epochs=epochs,seed=sd); accs.append(acc_of(q,p,Ete,yte))
    return np.array(accs),var
if RUN_WP1:
    print("== WP1: matched angle vs amplitude @8 feat, paired seeds =="); t0=time.time()
    ang8,_=run_matched(SEEDS_CORE,8,"angle"); amp8,_=run_matched(SEEDS_CORE,8,"amplitude")
    diff=amp8-ang8; m,sd_=diff.mean(),diff.std(ddof=1); n=len(diff)
    ci=stats.t.ppf(0.975,n-1)*sd_/np.sqrt(n); dz=m/sd_ if sd_>0 else float("nan"); tt=stats.ttest_rel(amp8,ang8)
    signs=np.array(list(itertools.product([1,-1],repeat=n))); pmean=(signs*np.abs(diff)).mean(1)
    p_perm=float(np.mean(np.abs(pmean)>=abs(m)))
    wp1=dict(seeds=SEEDS_CORE,angle8=ang8.tolist(),amp8=amp8.tolist(),diff=diff.tolist(),
             mean_diff=float(m),sd_diff=float(sd_),ci95=[float(m-ci),float(m+ci)],cohen_dz=float(dz),
             paired_t=float(tt.statistic),paired_p=float(tt.pvalue),signflip_p=p_perm,
             angle8_mean=float(ang8.mean()),angle8_sd=float(ang8.std(ddof=1)),
             amp8_mean=float(amp8.mean()),amp8_sd=float(amp8.std(ddof=1)))
    np.savez(os.path.join(RESULTS,"wp1_matched8.npz"),**{k:np.array(v) for k,v in wp1.items() if k!="seeds"})
    json.dump(wp1,open(os.path.join(RESULTS,"wp1_matched8.json"),"w"),indent=2)
    print(f"Angle-8 {ang8.mean():.3f}±{ang8.std(ddof=1):.3f} | Amp-8 {amp8.mean():.3f}±{amp8.std(ddof=1):.3f}")
    print(f"diff {m:+.4f}±{sd_:.4f}, 95%CI[{m-ci:+.4f},{m+ci:+.4f}], dz={dz:.2f}, t({n-1})={tt.statistic:.3f} p={tt.pvalue:.4f}, signflip p={p_perm:.4f}")
    print("WP1 %.1f min"%((time.time()-t0)/60))
if RUN_WP2:
    print("== WP2: angle-control family + amplitude sweep =="); rows=[]
    a8,v8=run_matched(SEEDS_SWEEP,8,"angle"); rows.append(["Angle (RY)",8,v8,a8.mean(),a8.std(ddof=1)])
    ad,vd=run_matched(SEEDS_SWEEP,16,"angle",dense=True); rows.append(["Angle-dense (RY+RZ)",16,vd,ad.mean(),ad.std(ddof=1)])
    for f in WP4_AMPFEAT:
        af,vf=run_matched(SEEDS_SWEEP,f,"amplitude"); rows.append(["Amplitude",f,vf,af.mean(),af.std(ddof=1)])
    dfm=pd.DataFrame(rows,columns=["Encoder","Features","Variance","Acc_mean","Acc_std"])
    savetab(dfm,"wp2_matched_sweep.csv"); print(dfm.to_string(index=False))
    amp=dfm[dfm.Encoder=="Amplitude"]; plt.figure(figsize=(7,4.3))
    plt.errorbar(amp.Features,amp.Acc_mean,yerr=amp.Acc_std.fillna(0),marker="o",capsize=3,label="Amplitude")
    a=dfm[dfm.Encoder=="Angle (RY)"]; plt.errorbar(a.Features,a.Acc_mean,yerr=a.Acc_std.fillna(0),fmt="s",capsize=3,label="Angle (RY), 8")
    d=dfm[dfm.Encoder.str.startswith("Angle-dense")]; plt.errorbar(d.Features,d.Acc_mean,yerr=d.Acc_std.fillna(0),fmt="^",capsize=3,label="Angle-dense (RY+RZ), 16")
    plt.xlabel("Features supplied to encoder"); plt.ylabel("11-class test accuracy")
    plt.title("Matched-encoder study (L=4)"); plt.legend(); plt.grid(alpha=.3); savefig("wp2_encoding_control.png")

== WP1: matched angle vs amplitude @8 feat, paired seeds ==
Angle-8 0.546±0.040 | Amp-8 0.571±0.032
diff +0.0252±0.0549, 95%CI[-0.0141,+0.0644], dz=0.46, t(9)=1.451 p=0.1807, signflip p=0.1992
WP1 19.6 min
== WP2: angle-control family + amplitude sweep ==
saved /kaggle/working/results/wp2_matched_sweep.csv
            Encoder  Features  Variance  Acc_mean  Acc_std
         Angle (RY)         8  0.787742  0.534848 0.035517
Angle-dense (RY+RZ)        16  0.929495  0.540000 0.058886
          Amplitude         8  0.787742  0.551212 0.020228
          Amplitude        16  0.929495  0.583182 0.025210
          Amplitude        32  0.997181  0.673030 0.010779
          Amplitude        64  1.000000  0.672121 0.010779
          Amplitude        79  1.000000  0.672121 0.010779
saved /kaggle/working/results/wp2_encoding_control.png


In [9]:
# ===================== WP4: cost & scaling =====================
def _count_1_2q(ops):
    g1=g2=0; st=list(ops)
    while st:
        op=st.pop(); nw=len(op.wires)
        if nw==1: g1+=1
        elif nw==2: g2+=1
        else:
            try: st.extend(op.decomposition())
            except Exception: pass
    return int(g1),int(g2)

def gate_counts(encoder,n_qubits,L,features,dense=False):
    kind="angle" if encoder=="angle" else "amplitude"
    Etr,_,_,_,_,_=prep(42,features,200,kind,n_qubits)
    dev=qml.device("default.qubit",wires=n_qubits)
    @qml.qnode(dev)
    def q(x,w):
        if encoder=="angle":
            qml.AngleEmbedding(x[:n_qubits],wires=range(n_qubits),rotation="Y")
            if dense: qml.AngleEmbedding(x[n_qubits:2*n_qubits],wires=range(n_qubits),rotation="Z")
        else: qml.AmplitudeEmbedding(x,wires=range(n_qubits),pad_with=0.0,normalize=True)
        qml.StronglyEntanglingLayers(w,wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
    w=0.1*np.random.default_rng(42).standard_normal(qml.StronglyEntanglingLayers.shape(n_layers=L,n_wires=n_qubits))
    tape=qml.workflow.construct_tape(q)(np.asarray(Etr[0]),w)
    return _count_1_2q(tape.operations)

if RUN_WP4:
    print("== WP4a: logical gate counts =="); rows=[]
    for enc,feat,dense,name in [("angle",8,False,"VQC/angle"),("angle",16,True,"angle-dense"),
                                ("amplitude",8,False,"Amplitude-8"),("amplitude",79,False,"Amplitude-79")]:
        g1,g2=gate_counts(enc,N_QUBITS,L_TEMPLATE,feat,dense=dense); rows.append([name,N_QUBITS,feat,g1,g2])
    dfg=pd.DataFrame(rows,columns=["Model","Qubits","Features","1q_gates","2q_gates"]); savetab(dfg,"wp4_gate_counts.csv"); print(dfg.to_string(index=False))

    print("== WP4b: runtime & memory scaling =="); rows=[]
    for nq in WP4_QUBITS:
        for L in WP4_DEPTHS:
            for ts in WP4_TRAIN:
                tracemalloc.start(); t0=time.time()
                Etr,Ete,ys,_,_,_=prep(42,nq,ts,"amplitude",nq)
                q,n_obs=matched_qnode("amplitude",nq,L); p=init_params(42,nq,L,n_obs)
                p=train_head_model(q,p,Etr,ys,epochs=max(4,EPOCHS//5),seed=42)
                dt=time.time()-t0; cur,peak=tracemalloc.get_traced_memory(); tracemalloc.stop()
                rows.append([nq,L,ts,round(dt,2),round(peak/1e6,1)]); print(f"  q={nq} L={L} N={ts}: {dt:.1f}s, {peak/1e6:.0f}MB")
    dfs=pd.DataFrame(rows,columns=["Qubits","Depth","TrainN","probe_seconds","peak_MB"]); savetab(dfs,"wp4_scaling.csv")
    plt.figure(figsize=(7,4.3))
    for nq in WP4_QUBITS:
        sub=dfs[(dfs.Qubits==nq)&(dfs.Depth==min(4,max(WP4_DEPTHS)))]
        if len(sub): plt.plot(sub.TrainN,sub.probe_seconds,marker="o",label=f"{nq} qubits")
    plt.xlabel("Training subsample"); plt.ylabel("probe train time (s)"); plt.title("WP4 runtime scaling")
    plt.legend(); plt.grid(alpha=.3); savefig("wp4_scaling.png")

    print("== WP4c: finite-shot reliability ==")
    Etr,Ete,ys,_,_,_=prep(42,79,TRAIN_SUB,"amplitude",N_QUBITS)
    qx,n_obs=matched_qnode("amplitude",N_QUBITS,6); p=init_params(42,N_QUBITS,6,n_obs)
    p=train_head_model(qx,p,Etr,ys,epochs=EPOCHS,seed=42); exact_acc=acc_of(qx,p,Ete,yte)
    Wamp=np_w(p); Ah,bh=np_head(p)
    obs=[qml.PauliZ(i) for i in range(N_QUBITS)]+[qml.PauliZ(i)@qml.PauliZ((i+1)%N_QUBITS) for i in range(N_QUBITS)]
    sub=np.random.default_rng(0).permutation(len(Ete))[:300]; rows=[]
    for shots in WP4_SHOTS:
        devs=qml.device("default.qubit",wires=N_QUBITS,shots=shots)
        @qml.qnode(devs)
        def qs(x,w):
            qml.AmplitudeEmbedding(x,wires=range(N_QUBITS),pad_with=0.0,normalize=True)
            qml.StronglyEntanglingLayers(w,wires=range(N_QUBITS))
            return [qml.expval(o) for o in obs]
        accs=[]
        for r in range(3 if QUICK_TEST else 10):
            e=np.stack([qs(np.asarray(Ete[i]),Wamp) for i in sub]); logits=(Ah@e.T).T+bh
            accs.append(accuracy_score(yte[sub],logits.argmax(1)))
        rows.append([shots,np.mean(accs),np.std(accs),np.mean(accs)-exact_acc]); print(f"  shots={shots}: {np.mean(accs):.3f}±{np.std(accs):.3f} (exact {exact_acc:.3f})")
    savetab(pd.DataFrame(rows,columns=["shots","acc_mean","acc_std","diff_from_exact"]),"wp4_finite_shots.csv")


== WP4a: logical gate counts ==
saved /kaggle/working/results/wp4_gate_counts.csv
       Model  Qubits  Features  1q_gates  2q_gates
   VQC/angle       8         8        40        32
 angle-dense       8        16        48        32
 Amplitude-8       8         8       511       540
Amplitude-79       8        79       489       540
== WP4b: runtime & memory scaling ==
  q=6 L=2 N=450: 1.6s, 5MB
  q=6 L=2 N=900: 3.0s, 6MB
  q=6 L=2 N=1800: 5.9s, 7MB
  q=6 L=4 N=450: 2.4s, 5MB
  q=6 L=4 N=900: 4.7s, 6MB
  q=6 L=4 N=1800: 9.3s, 7MB
  q=6 L=6 N=450: 3.2s, 5MB
  q=6 L=6 N=900: 6.3s, 6MB
  q=6 L=6 N=1800: 12.5s, 7MB
  q=8 L=2 N=450: 2.5s, 12MB
  q=8 L=2 N=900: 5.0s, 14MB
  q=8 L=2 N=1800: 10.2s, 17MB
  q=8 L=4 N=450: 4.3s, 12MB
  q=8 L=4 N=900: 8.5s, 14MB
  q=8 L=4 N=1800: 16.1s, 17MB
  q=8 L=6 N=450: 5.7s, 12MB
  q=8 L=6 N=900: 11.0s, 14MB
  q=8 L=6 N=1800: 22.6s, 17MB
  q=10 L=2 N=450: 6.4s, 42MB
  q=10 L=2 N=900: 12.2s, 46MB
  q=10 L=2 N=1800: 24.5s, 55MB
  q=10 L=4 N=450: 9.3s, 42MB
 

In [10]:
# ===================== WP5: noise (density matrix, CPU) =====================
if RUN_WP5:
    print("== WP5: noise channels (evaluate fixed trained model) ==")
    Etr,Ete,ys,_,_,_=prep(42,N_QUBITS,TRAIN_SUB,"angle",N_QUBITS)
    q,n_obs=matched_qnode("angle",N_QUBITS,L_TEMPLATE); p=init_params(42,N_QUBITS,L_TEMPLATE,n_obs)
    p=train_head_model(q,p,Etr,ys,epochs=EPOCHS,seed=42); clean=acc_of(q,p,Ete,yte)
    Wtr=np_w(p); Ah,bh=np_head(p)
    obs=[qml.PauliZ(i) for i in range(N_QUBITS)]+[qml.PauliZ(i)@qml.PauliZ((i+1)%N_QUBITS) for i in range(N_QUBITS)]
    NOISE_SUB=120 if QUICK_TEST else 200
    sub=np.random.default_rng(1).permutation(len(Ete))[:NOISE_SUB]
    def noisy_acc(channel,prob,readout=0.0):
        dev=qml.device("default.mixed",wires=N_QUBITS)
        @qml.qnode(dev)
        def qn(x):
            qml.AngleEmbedding(x[:N_QUBITS],wires=range(N_QUBITS),rotation="Y")
            qml.StronglyEntanglingLayers(Wtr,wires=range(N_QUBITS))
            if channel:
                for w_ in range(N_QUBITS): channel(prob,wires=w_)
            if readout>0:
                for w_ in range(N_QUBITS): qml.BitFlip(readout,wires=w_)
            return [qml.expval(o) for o in obs]
        e=np.stack([qn(np.asarray(Ete[i])) for i in sub]); logits=(Ah@e.T).T+bh
        return accuracy_score(yte[sub],logits.argmax(1))
    rows=[["clean",0.0,clean]]
    for cname,ch in {"depolarizing":qml.DepolarizingChannel,"amplitude_damping":qml.AmplitudeDamping,"phase_damping":qml.PhaseDamping}.items():
        for lvl,pr in WP5_LEVELS.items():
            a=noisy_acc(ch,pr); rows.append([cname,pr,a]); print(f"  {cname} p={pr}: {a:.3f}")
    for lvl,pr in WP5_LEVELS.items():
        a=noisy_acc(None,0.0,readout=pr); rows.append(["readout_error",pr,a]); print(f"  readout p={pr}: {a:.3f}")
    def combined():
        dev=qml.device("default.mixed",wires=N_QUBITS)
        @qml.qnode(dev)
        def qc(x):
            qml.AngleEmbedding(x[:N_QUBITS],wires=range(N_QUBITS),rotation="Y")
            qml.StronglyEntanglingLayers(Wtr,wires=range(N_QUBITS))
            for w_ in range(N_QUBITS):
                qml.DepolarizingChannel(WP5_LEVELS["moderate"],wires=w_)
                qml.AmplitudeDamping(WP5_LEVELS["low"],wires=w_)
                qml.BitFlip(WP5_LEVELS["moderate"],wires=w_)
            return [qml.expval(o) for o in obs]
        e=np.stack([qc(np.asarray(Ete[i])) for i in sub]); logits=(Ah@e.T).T+bh
        return accuracy_score(yte[sub],logits.argmax(1))
    ac=combined(); rows.append(["combined_moderate",WP5_LEVELS["moderate"],ac]); print(f"  combined: {ac:.3f}")
    savetab(pd.DataFrame(rows,columns=["noise","level","accuracy"]),"wp5_noise.csv")
    if WP5_TRANSPILE:
        try:
            pipq("pennylane-qiskit","qiskit"); from qiskit import transpile
            print("  (transpile enabled — fill swap/depth from transpile(...).count_ops() as needed)")
        except Exception as e: print("  transpile skipped:",e)


== WP5: noise channels (evaluate fixed trained model) ==
  depolarizing p=0.005: 0.630
  depolarizing p=0.02: 0.640
  depolarizing p=0.05: 0.640
  amplitude_damping p=0.005: 0.650
  amplitude_damping p=0.02: 0.565
  amplitude_damping p=0.05: 0.550
  phase_damping p=0.005: 0.630
  phase_damping p=0.02: 0.630
  phase_damping p=0.05: 0.630
  readout p=0.005: 0.620
  readout p=0.02: 0.640
  readout p=0.05: 0.645
  combined: 0.640
saved /kaggle/working/results/wp5_noise.csv


In [11]:
# ===================== WP6: multi-seed FGSM + drift =====================
def classical_mlp(seed,n_pca=N_QUBITS,train_sub=TRAIN_SUB):
    Etr,Ete,ys,ybin,_,_=prep(seed,n_pca,train_sub,"angle",N_QUBITS)
    clf=MLPClassifier(hidden_layer_sizes=(64,),max_iter=400,random_state=seed).fit(Etr,ys)
    return clf,Etr,Ete,ys

if RUN_WP6:
    print("== WP6: FGSM across seeds (quantum vs classical) ==")
    eps_grid=[0.0,0.05,0.10,0.15,0.20,0.30,0.40]; qs_all=[]; cs_all=[]
    for sd in SEEDS_SWEEP:
        Etr,Ete,ys,_,_,_=prep(sd,N_QUBITS,TRAIN_SUB,"angle",N_QUBITS)
        q,n_obs=matched_qnode("angle",N_QUBITS,L_TEMPLATE); p=init_params(sd,N_QUBITS,L_TEMPLATE,n_obs)
        p=train_head_model(q,p,Etr,ys,epochs=EPOCHS,seed=sd); w,A,b=p
        clf,_,_,_=classical_mlp(sd)
        subte=np.random.default_rng(sd).permutation(len(Ete))[:300]; ye=yte[subte]
        Xe=torch.as_tensor(Ete[subte],dtype=TDT,device=TDEV).requires_grad_(True)
        e=torch.stack(q(Xe,w)); logits=(A@e).T+b
        loss=torch.nn.functional.cross_entropy(logits,torch.as_tensor(ye,dtype=torch.long,device=TDEV))
        loss.backward(); gsign=Xe.grad.sign().detach().cpu().numpy(); base=Ete[subte]
        qc=[]; cc=[]
        for eps in eps_grid:
            Xadv=np.clip(base+eps*gsign,0,np.pi)
            qc.append(acc_of(q,p,Xadv,ye)); cc.append(accuracy_score(ye,clf.predict(Xadv)))
        qs_all.append(qc); cs_all.append(cc); print(f"  seed {sd} done")
    qs_all=np.array(qs_all); cs_all=np.array(cs_all); ms=len(SEEDS_SWEEP)>1
    dffg=pd.DataFrame({"epsilon":eps_grid,"quantum_mean":qs_all.mean(0),
        "quantum_std":qs_all.std(0,ddof=1) if ms else np.zeros(len(eps_grid)),
        "classical_mean":cs_all.mean(0),"classical_std":cs_all.std(0,ddof=1) if ms else np.zeros(len(eps_grid))})
    savetab(dffg,"wp6_fgsm.csv"); print(dffg.to_string(index=False))
    plt.figure(figsize=(7,4.3))
    plt.errorbar(eps_grid,dffg.quantum_mean,yerr=dffg.quantum_std,marker="o",capsize=3,label="Quantum QNN-DRU")
    plt.errorbar(eps_grid,dffg.classical_mean,yerr=dffg.classical_std,marker="s",capsize=3,label="Classical MLP (PCA-8)")
    plt.xlabel("FGSM budget epsilon"); plt.ylabel("accuracy"); plt.title("WP6 adversarial robustness (multi-seed)")
    plt.legend(); plt.grid(alpha=.3); savefig("wp6_fgsm.png")

    print("== WP6: synthetic drift ==")
    Etr,Ete,ys,_,_,_=prep(42,N_QUBITS,TRAIN_SUB,"angle",N_QUBITS)
    q,n_obs=matched_qnode("angle",N_QUBITS,L_TEMPLATE); p=init_params(42,N_QUBITS,L_TEMPLATE,n_obs)
    p=train_head_model(q,p,Etr,ys,epochs=EPOCHS,seed=42)
    win=200; ddir=np.zeros(N_QUBITS); ddir[0]=1.0; rows=[]
    for w0 in range(0,len(Ete)-win,win):
        alpha=0.8*(w0/len(Ete)); Ew=np.clip(Ete[w0:w0+win]+alpha*np.pi*ddir,0,np.pi)
        rows.append([w0,alpha,acc_of(q,p,Ew,yte[w0:w0+win])])
    savetab(pd.DataFrame(rows,columns=["flow_index","drift_alpha","windowed_acc"]),"wp6_drift.csv")


== WP6: FGSM across seeds (quantum vs classical) ==
  seed 42 done
  seed 7 done
  seed 123 done
saved /kaggle/working/results/wp6_fgsm.csv
 epsilon  quantum_mean  quantum_std  classical_mean  classical_std
    0.00      0.531111     0.075006        0.605556       0.041410
    0.05      0.271111     0.037908        0.367778       0.074932
    0.10      0.200000     0.059255        0.327778       0.094771
    0.15      0.158889     0.061494        0.262222       0.087008
    0.20      0.142222     0.045256        0.215556       0.088464
    0.30      0.096667     0.045826        0.164444       0.109358
    0.40      0.032222     0.010715        0.143333       0.052387
saved /kaggle/working/results/wp6_fgsm.png
== WP6: synthetic drift ==
saved /kaggle/working/results/wp6_drift.csv


In [12]:
# ===================== WP3: dedup + repeated stratified splits =====================
if RUN_WP3:
    print("== WP3: remove exact overlaps, repeated stratified splits ==")
    Xall=np.vstack([Xtr_raw,Xte_raw]); yall=np.concatenate([ytr,yte])
    _,uniq=np.unique(np.round(Xall,6),axis=0,return_index=True); uniq=np.sort(uniq)
    Xc,yc=Xall[uniq],yall[uniq]; print(f"  pooled {len(Xall)} -> {len(Xc)} after dedup")
    from sklearn.model_selection import StratifiedShuffleSplit
    sss=StratifiedShuffleSplit(n_splits=WP3_N_SPLITS,test_size=WP3_TEST_FRAC,random_state=42); rows=[]
    for si,(tri,tei) in enumerate(sss.split(Xc,yc)):
        sc=StandardScaler().fit(Xc[tri]); Xtr_s,Xte_s=sc.transform(Xc[tri]),sc.transform(Xc[tei])
        pca=PCA(n_components=N_QUBITS,random_state=42).fit(Xtr_s); Ztr,Zte=pca.transform(Xtr_s),pca.transform(Xte_s)
        zmin,zmax=Ztr.min(0),Ztr.max(0); den=(zmax-zmin)+1e-9; Atr,Ate=np.pi*(Ztr-zmin)/den,np.pi*(Zte-zmin)/den
        sub=np.random.default_rng(si).permutation(len(Atr))[:TRAIN_SUB]
        q,n_obs=matched_qnode("angle",N_QUBITS,L_TEMPLATE); pr=init_params(si,N_QUBITS,L_TEMPLATE,n_obs)
        pr=train_head_model(q,pr,Atr[sub],yc[tri][sub],epochs=EPOCHS,seed=si); qacc=acc_of(q,pr,Ate,yc[tei])
        mlp=MLPClassifier(hidden_layer_sizes=(64,),max_iter=400,random_state=si).fit(Atr[sub],yc[tri][sub]); macc=accuracy_score(yc[tei],mlp.predict(Ate))
        lr=LogisticRegression(max_iter=500).fit(Atr[sub],yc[tri][sub]); lacc=accuracy_score(yc[tei],lr.predict(Ate))
        rows.append([si,qacc,macc,lacc]); print(f"  split {si}: Q={qacc:.3f} MLP={macc:.3f} LR={lacc:.3f}")
    dfw3=pd.DataFrame(rows,columns=["split","quantum_acc","mlp_acc","logreg_acc"]); savetab(dfw3,"wp3_dedup_splits.csv")
    summ=dfw3[["quantum_acc","mlp_acc","logreg_acc"]].agg(["mean","std"]); print(summ.to_string())
    json.dump({"mean":summ.loc["mean"].to_dict(),"std":summ.loc["std"].to_dict()},open(os.path.join(RESULTS,"wp3_summary.json"),"w"),indent=2)


== WP3: remove exact overlaps, repeated stratified splits ==
  pooled 7326 -> 6786 after dedup
  split 0: Q=0.459 MLP=0.578 LR=0.553
  split 1: Q=0.490 MLP=0.634 LR=0.456
  split 2: Q=0.446 MLP=0.572 LR=0.449
  split 3: Q=0.533 MLP=0.632 LR=0.495
  split 4: Q=0.453 MLP=0.500 LR=0.484
saved /kaggle/working/results/wp3_dedup_splits.csv
      quantum_acc   mlp_acc  logreg_acc
mean     0.476424  0.583104    0.487230
std      0.035727  0.054507    0.041189


In [13]:
# ===================== DATA AUDIT (Sec 4) + PCA curve (Fig 13) =====================
if True:
    print("== Data audit ==")
    def dup_count(X):
        _,c=np.unique(np.round(X,6),axis=0,return_counts=True); return int((c>1).sum()), int((c[c>1]-1).sum())
    tr_groups,tr_extra=dup_count(Xtr_raw); te_groups,te_extra=dup_count(Xte_raw)
    keyset=set(map(lambda r:tuple(np.round(r,6)),Xtr_raw))
    cross=sum(1 for r in Xte_raw if tuple(np.round(r,6)) in keyset)
    const_cols=int((Xtr_raw.std(0)==0).sum())
    audit=dict(train_dup_rows=tr_extra,test_dup_rows=te_extra,cross_overlap=cross,
               cross_pct=round(100*cross/len(Xte_raw),1),constant_cols=const_cols,
               n_train=len(Xtr_raw),n_test=len(Xte_raw))
    json.dump(audit,open(os.path.join(RESULTS,"data_audit.json"),"w"),indent=2); print(audit)
    # PCA variance retention vs qubits (Fig 13)
    sc=StandardScaler().fit(Xtr_raw); Xs=sc.transform(Xtr_raw); rows=[]
    for q in range(4,13):
        pca=PCA(n_components=q,random_state=42).fit(Xs); rows.append([q,float(pca.explained_variance_ratio_.sum())])
    dfv=pd.DataFrame(rows,columns=["qubits","retained_variance"]); savetab(dfv,"pca_variance.csv")
    plt.figure(figsize=(7,4)); plt.plot(dfv.qubits,dfv.retained_variance*100,marker="o")
    plt.axvline(8,ls="--",c="r",alpha=.6); plt.xlabel("qubits (PCA components)"); plt.ylabel("retained variance (%)")
    plt.title("PCA variance retention (smooth, no power-of-two effect)"); plt.grid(alpha=.3); savefig("fig_pca_variance.png")


== Data audit ==
{'train_dup_rows': 340, 'test_dup_rows': 99, 'cross_overlap': 200, 'cross_pct': 9.1, 'constant_cols': 11, 'n_train': 5126, 'n_test': 2200}
saved /kaggle/working/results/pca_variance.csv
saved /kaggle/working/results/fig_pca_variance.png


In [15]:
# ===================== MODEL ZOO: VQC, QCNN, QSVM (+ generic binary) =====================
def vqc_train_eval(seed,epochs=EPOCHS,sub=TRAIN_SUB,binary=False):
    Etr,Ete,ys,ybs,_,_=prep(seed,N_QUBITS,sub,"angle",N_QUBITS)
    dev=qml.device("default.qubit",wires=N_QUBITS)
    @qml.qnode(dev,interface="torch",diff_method="backprop")
    def vqc(x,w):
        qml.AngleEmbedding(x,wires=range(N_QUBITS),rotation="Y"); qml.StronglyEntanglingLayers(w,wires=range(N_QUBITS))
        return qml.probs(wires=[0,1,2,3])
    g=torch.Generator().manual_seed(int(seed)); w=(0.1*torch.randn(qml.StronglyEntanglingLayers.shape(3,N_QUBITS),generator=g,dtype=TDT)).to(TDEV).detach().requires_grad_(True)
    y=torch.as_tensor(ybs if binary else ys,dtype=torch.long,device=TDEV); C=2 if binary else 11
    Xt=torch.as_tensor(Etr,dtype=TDT,device=TDEV); opt=torch.optim.Adam([w],0.05)
    for ep in range(epochs):
        p=vqc(Xt,w)[:, :C]; p=p/p.sum(1,keepdim=True)
        loss=torch.nn.functional.nll_loss(torch.log(p+1e-9),y); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pt=vqc(torch.as_tensor(Ete,dtype=TDT,device=TDEV),w)[:, :C].cpu().numpy()
    pt=pt/pt.sum(1,keepdims=True)
    if binary: return roc_auc_score(ybin_te,pt[:,1]), accuracy_score(ybin_te,pt.argmax(1))
    return accuracy_score(yte,pt.argmax(1)), f1_score(yte,pt.argmax(1),average="weighted"), pt.argmax(1)

def _conv(p,wires):
    qml.RY(p[0],wires=wires[0]); qml.RY(p[1],wires=wires[1]); qml.CNOT(wires=wires)
    qml.RY(p[2],wires=wires[0]); qml.RY(p[3],wires=wires[1]); qml.CNOT(wires=wires[::-1])
    qml.RZ(p[4],wires=wires[0]); qml.RZ(p[5],wires=wires[1])
def qcnn_train_eval(seed,epochs=EPOCHS,sub=TRAIN_SUB,binary=False):
    Etr,Ete,ys,ybs,_,_=prep(seed,N_QUBITS,sub,"angle",N_QUBITS)
    dev=qml.device("default.qubit",wires=N_QUBITS); pairs=[(0,1),(2,3),(4,5),(6,7),(1,3),(5,7),(3,7)]
    @qml.qnode(dev,interface="torch",diff_method="backprop")
    def qcnn(x,w):
        qml.AngleEmbedding(x,wires=range(N_QUBITS),rotation="Y"); i=0
        for a,b in pairs: _conv(w[i:i+6],[a,b]); i+=6
        return [qml.expval(qml.PauliZ(3)),qml.expval(qml.PauliZ(7)),qml.expval(qml.PauliZ(3)@qml.PauliZ(7))]
    C=2 if binary else 11; g=torch.Generator().manual_seed(int(seed))
    w=(0.1*torch.randn(6*len(pairs),generator=g,dtype=TDT)).to(TDEV).detach().requires_grad_(True)
    A=(0.1*torch.randn(C,3,generator=g,dtype=TDT)).to(TDEV).detach().requires_grad_(True); b=torch.zeros(C,dtype=TDT,device=TDEV,requires_grad=True)
    Xt=torch.as_tensor(Etr,dtype=TDT,device=TDEV); y=torch.as_tensor(ybs if binary else ys,dtype=torch.long,device=TDEV)
    opt=torch.optim.Adam([w,A,b],0.03)
    for ep in range(epochs):
        e=torch.stack(qcnn(Xt,w)); logits=(A@e).T+b
        loss=torch.nn.functional.cross_entropy(logits,y); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        e=torch.stack(qcnn(torch.as_tensor(Ete,dtype=TDT,device=TDEV),w)); sm=torch.softmax((A@e).T+b,1).cpu().numpy()
    if binary: return roc_auc_score(ybin_te,sm[:,1]), accuracy_score(ybin_te,sm.argmax(1))
    return accuracy_score(yte,sm.argmax(1)), f1_score(yte,sm.argmax(1),average="weighted"), sm.argmax(1)

def qsvm_train_eval(seed,Ntr=None,Nte=None,nk=6,binary=False):
    Ntr=Ntr or QSVM_NTR; Nte=Nte or QSVM_NTE
    Etr,Ete,ys,ybs,_,_=prep(seed,nk,max(Ntr,TRAIN_SUB),"angle",nk)
    devk=qml.device("default.qubit",wires=nk)
    @qml.qnode(devk)
    def kern(x1,x2):
        qml.AngleEmbedding(x1[:nk],wires=range(nk),rotation="Y"); qml.adjoint(qml.AngleEmbedding)(x2[:nk],wires=range(nk),rotation="Y")
        return qml.probs(wires=range(nk))
    def gram(Xa,Xb):
        G=np.zeros((len(Xa),len(Xb)))
        for i in range(len(Xa)):
            for j in range(len(Xb)): G[i,j]=kern(np.asarray(Xa[i]),np.asarray(Xb[j]))[0]
        return G
    As=Etr[:Ntr]; lab=(ybs if binary else ys)[:Ntr]; ss=min(Nte,len(Ete)); Bs=Ete[:ss]
    Gtr=gram(As,As); svc=SVC(kernel="precomputed",probability=binary).fit(Gtr,lab); Gte=gram(Bs,As)
    ytrue=(ybin_te if binary else yte)[:ss]
    if binary:
        try: sc=svc.predict_proba(Gte)[:,1]
        except Exception: sc=svc.decision_function(Gte)
        return roc_auc_score(ytrue,sc), accuracy_score(ytrue,svc.predict(Gte)), int(svc.n_support_.sum())
    pred=svc.predict(Gte); return accuracy_score(ytrue,pred), f1_score(ytrue,pred,average="weighted"), int(svc.n_support_.sum())
print("model zoo ready (VQC / QCNN / QSVM).")


model zoo ready (VQC / QCNN / QSVM).


In [16]:
# ===================== 11-CLASS CASE STUDIES (Tables 5,6; Fig 5) =====================
if True:
    print("== 11-class case studies (VQC/QCNN/QSVM/QNN-DRU/Amplitude) ==")
    QSEEDS = SEEDS_SWEEP
    # QNN-DRU (angle) and Amplitude via matched template
    def tmpl_eval(seed,encoder,features):
        Etr,Ete,ys,_,_,_=prep(seed,features,TRAIN_SUB,"angle" if encoder=="angle" else "amplitude",N_QUBITS)
        q,n_obs=matched_qnode(encoder,N_QUBITS,L_TEMPLATE); p=init_params(seed,N_QUBITS,L_TEMPLATE,n_obs)
        p=train_head_model(q,p,Etr,ys,epochs=EPOCHS,seed=seed); pred=predict_head(q,p,Ete).argmax(1)
        return accuracy_score(yte,pred), f1_score(yte,pred,average="weighted"), pred
    # seed-42 point estimates (Table 5)
    vqc_a=vqc_train_eval(42); qcnn_a=qcnn_train_eval(42); qsvm_a=qsvm_train_eval(42)
    dru_a=tmpl_eval(42,"angle",8); amp_a=tmpl_eval(42,"amplitude",79)
    t5=pd.DataFrame([["VQC",*vqc_a[:2]],["QCNN",*qcnn_a[:2]],["QSVM(subsample)",*qsvm_a[:2]],
                     ["QNN-DRU",*dru_a[:2]],["Amplitude",*amp_a[:2]]],columns=["Model","Accuracy","F1w"])
    savetab(t5,"table5_11class_seed42.csv"); print(t5.to_string(index=False))
    # 3-seed means (Table 6)
    def mean_std(fn):
        accs=[]; f1s=[]
        for s in QSEEDS:
            r=fn(s); accs.append(r[0]); f1s.append(r[1])
        return np.mean(accs),np.std(accs,ddof=1) if len(accs)>1 else 0.0,np.mean(f1s),np.std(f1s,ddof=1) if len(f1s)>1 else 0.0
    rows=[]
    for name,fn in [("VQC",lambda s:vqc_train_eval(s)),("QCNN",lambda s:qcnn_train_eval(s)),
                    ("QNN-DRU",lambda s:tmpl_eval(s,"angle",8)),("Amplitude",lambda s:tmpl_eval(s,"amplitude",79))]:
        m,sd,fm,fsd=mean_std(fn); rows.append([name,m,sd,fm,fsd]); print(f"  {name}: acc {m:.3f}±{sd:.3f}")
    t6=pd.DataFrame(rows,columns=["Model","Acc_mean","Acc_std","F1w_mean","F1w_std"]); savetab(t6,"table6_11class_3seed.csv")
    # Fig 5 bar
    plt.figure(figsize=(7,4.3)); x=np.arange(len(t5)); plt.bar(x-0.2,t5.Accuracy,0.4,label="Accuracy"); plt.bar(x+0.2,t5.F1w,0.4,label="Weighted F1")
    plt.axhline(1/11,ls="--",c="gray"); plt.xticks(x,t5.Model,rotation=20); plt.ylabel("Score"); plt.title("11-class attack-type identification (seed 42)")
    plt.legend(); plt.grid(alpha=.3,axis="y"); savefig("fig5_11class.png")
    _AMP_PRED=amp_a[2]; _DRU_PRED=dru_a[2]  # for confusion matrices


== 11-class case studies (VQC/QCNN/QSVM/QNN-DRU/Amplitude) ==
saved /kaggle/working/results/table5_11class_seed42.csv
          Model  Accuracy      F1w
            VQC  0.277727 0.179733
           QCNN  0.352273 0.247681
QSVM(subsample)  0.410714 0.325714
        QNN-DRU  0.566364 0.501931
      Amplitude  0.669091 0.647921
  VQC: acc 0.312±0.081
  QCNN: acc 0.335±0.025
  QNN-DRU: acc 0.535±0.036
  Amplitude: acc 0.672±0.011
saved /kaggle/working/results/table6_11class_3seed.csv
saved /kaggle/working/results/fig5_11class.png


In [17]:
# ===================== CONFUSION + MACRO/PER-CLASS (Figs 6,7,8; Table 7) =====================
if True:
    from sklearn.metrics import precision_score, recall_score
    names=[c for c in CLASSES]
    def plot_cm(pred,title,fname):
        cm=confusion_matrix(yte,pred,labels=range(len(CLASSES))).astype(float)
        cm=cm/cm.sum(1,keepdims=True).clip(min=1)
        plt.figure(figsize=(7,6)); plt.imshow(cm,cmap="viridis",vmin=0,vmax=1)
        plt.xticks(range(len(names)),names,rotation=90,fontsize=7); plt.yticks(range(len(names)),names,fontsize=7)
        plt.colorbar(); plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True"); savefig(fname)
    plot_cm(_AMP_PRED,"Amplitude QNN confusion (row-normalised)","fig6_amp_confusion.png")
    plot_cm(_DRU_PRED,"QNN-DRU confusion (row-normalised)","fig7_dru_confusion.png")
    # macro / per-class for amplitude (Table 7)
    macro=dict(accuracy=accuracy_score(yte,_AMP_PRED),balanced_acc=balanced_accuracy_score(yte,_AMP_PRED),
               macro_P=precision_score(yte,_AMP_PRED,average="macro",zero_division=0),
               macro_R=recall_score(yte,_AMP_PRED,average="macro",zero_division=0),
               macro_F1=f1_score(yte,_AMP_PRED,average="macro",zero_division=0),
               F1w=f1_score(yte,_AMP_PRED,average="weighted",zero_division=0))
    per_recall={CLASSES[i]:float(recall_score(yte==i,_AMP_PRED==i,zero_division=0)) for i in range(len(CLASSES))}
    json.dump({"macro":macro,"per_class_recall":per_recall},open(os.path.join(RESULTS,"table7_macro.json"),"w"),indent=2)
    print("Table 7 macro (amplitude):",{k:round(v,3) for k,v in macro.items()})
    # 5-class operational aggregation (Fig 8)
    CAT5={"NORMAL":"Normal","DNP3_ENUMERATE":"Recon","DNP3_INFO":"Recon","ARP_POISONING":"MITM","MITM_DOS":"MITM",
          "COLD_RESTART":"Func_Abuse","WARM_RESTART":"Func_Abuse","STOP_APP":"Func_Abuse","INIT_DATA":"Func_Abuse",
          "DISABLE_UNSOLICITED":"Func_Abuse","REPLAY":"Replay"}
    order=["Normal","Recon","MITM","Func_Abuse","Replay"]; c5={c:i for i,c in enumerate(order)}
    def to5(idx_arr): return np.array([c5[CAT5.get(CLASSES[i],"Normal")] for i in idx_arr])
    yte5=to5(yte); pred5=to5(_AMP_PRED)
    cm5=confusion_matrix(yte5,pred5,labels=range(5)).astype(float); cm5=cm5/cm5.sum(1,keepdims=True).clip(min=1)
    plt.figure(figsize=(6,5)); plt.imshow(cm5,cmap="viridis",vmin=0,vmax=1)
    plt.xticks(range(5),order,rotation=30); plt.yticks(range(5),order); plt.colorbar()
    plt.title("Amplitude QNN 5-class (aggregated)"); plt.xlabel("Predicted"); plt.ylabel("True"); savefig("fig8_5class_confusion.png")
    json.dump({"acc":float(accuracy_score(yte5,pred5)),"balanced_acc":float(balanced_accuracy_score(yte5,pred5)),
               "macro_F1":float(f1_score(yte5,pred5,average="macro"))},open(os.path.join(RESULTS,"table7_5class.json"),"w"),indent=2)


saved /kaggle/working/results/fig6_amp_confusion.png
saved /kaggle/working/results/fig7_dru_confusion.png
Table 7 macro (amplitude): {'accuracy': 0.669, 'balanced_acc': np.float64(0.669), 'macro_P': 0.71, 'macro_R': 0.669, 'macro_F1': 0.648, 'F1w': 0.648}
saved /kaggle/working/results/fig8_5class_confusion.png


In [18]:
# ===================== BINARY DETECTION + HIERARCHICAL BOOTSTRAP (C2; Tables 11; Figs 10,11) =====================
if True:
    print("== Binary detection: 4 models (seed 42) ==")
    b_vqc=vqc_train_eval(42,binary=True); b_qcnn=qcnn_train_eval(42,binary=True); b_qsvm=qsvm_train_eval(42,binary=True)
    # QNN-DRU binary (matched template, C=2)
    def dru_binary(seed,sub=TRAIN_SUB,epochs=EPOCHS):
        Etr,Ete,ys,ybs,_,_=prep(seed,N_QUBITS,sub,"angle",N_QUBITS)
        q,n_obs=matched_qnode("angle",N_QUBITS,L_TEMPLATE); p=init_params(seed,N_QUBITS,L_TEMPLATE,n_obs,C=2)
        p=train_head_model(q,p,Etr,ybs,epochs=epochs,seed=seed)
        logits=predict_head(q,p,Ete); sm=np.exp(logits-logits.max(1,keepdims=True)); sm/=sm.sum(1,keepdims=True)
        return roc_auc_score(ybin_te,sm[:,1]), accuracy_score(ybin_te,sm.argmax(1)), sm[:,1]
    b_dru=dru_binary(42)
    t11=pd.DataFrame([["VQC",b_vqc[1],b_vqc[0]],["QCNN",b_qcnn[1],b_qcnn[0]],
                      ["QSVM(subsample)",b_qsvm[1],b_qsvm[0]],["QNN-DRU",b_dru[1],b_dru[0]]],
                     columns=["Model","Accuracy","AUC"]); savetab(t11,"table11_binary_seed42.csv"); print(t11.to_string(index=False))

    print("== Hierarchical bootstrap: QNN-DRU vs MLP AUC (C2) ==")
    def mlp_binary(seed,sub=TRAIN_SUB):
        Etr,Ete,ys,ybs,_,_=prep(seed,N_QUBITS,sub,"angle",N_QUBITS)
        clf=MLPClassifier(hidden_layer_sizes=(64,),max_iter=400,random_state=seed).fit(Etr,ybs)
        sc=clf.predict_proba(Ete)[:,1]; return roc_auc_score(ybin_te,sc), sc
    q_auc=[]; m_auc=[]; q_sc=[]; m_sc=[]
    for s in SEEDS_BINARY:
        qa,_,qs_=dru_binary(s); ma,ms_=mlp_binary(s); q_auc.append(qa); m_auc.append(ma); q_sc.append(qs_); m_sc.append(ms_)
        print(f"  seed {s}: QNN {qa:.3f} vs MLP {ma:.3f}  (diff {qa-ma:+.3f})")
    q_auc=np.array(q_auc); m_auc=np.array(m_auc); diffs=q_auc-m_auc
    # hierarchical bootstrap: resample seeds then test cases
    rng=np.random.default_rng(0); B=2000; nte=len(ybin_te); boot=[]
    for _ in range(B):
        si=rng.integers(0,len(SEEDS_BINARY),len(SEEDS_BINARY)); ti=rng.integers(0,nte,nte)
        d=np.mean([roc_auc_score(ybin_te[ti],q_sc[k][ti])-roc_auc_score(ybin_te[ti],m_sc[k][ti]) for k in si])
        boot.append(d)
    boot=np.array(boot); lo,hi=np.percentile(boot,[2.5,97.5]); p_boot=float(2*min((boot<=0).mean(),(boot>=0).mean()))
    res=dict(seeds=SEEDS_BINARY,qnn_auc=q_auc.tolist(),mlp_auc=m_auc.tolist(),per_seed_diff=diffs.tolist(),
             mean_diff=float(diffs.mean()),ci95=[float(lo),float(hi)],p_boot=p_boot,
             qnn_mean=float(q_auc.mean()),qnn_sd=float(q_auc.std(ddof=1)),mlp_mean=float(m_auc.mean()),mlp_sd=float(m_auc.std(ddof=1)))
    json.dump(res,open(os.path.join(RESULTS,"binary_hier_bootstrap.json"),"w"),indent=2)
    print(f"  mean diff {diffs.mean():+.4f}, 95% hier-CI [{lo:+.4f},{hi:+.4f}], p={p_boot:.3f}")
    plt.figure(figsize=(7,4)); x=np.arange(len(SEEDS_BINARY))
    plt.bar(x-0.2,q_auc,0.4,label="QNN-DRU"); plt.bar(x+0.2,m_auc,0.4,label="Classical MLP")
    plt.xticks(x,[f"seed {s}" for s in SEEDS_BINARY],rotation=20); plt.ylim(0.8,1.0); plt.ylabel("Binary ROC-AUC")
    plt.title("Per-seed binary AUC (both retrained)"); plt.legend(); plt.grid(alpha=.3,axis="y"); savefig("fig11_binary_auc.png")


== Binary detection: 4 models (seed 42) ==
saved /kaggle/working/results/table11_binary_seed42.csv
          Model  Accuracy      AUC
            VQC  0.908636 0.879293
           QCNN  0.909091 0.896007
QSVM(subsample)  0.885714 0.944682
        QNN-DRU  0.950000 0.918348
== Hierarchical bootstrap: QNN-DRU vs MLP AUC (C2) ==
  seed 42: QNN 0.918 vs MLP 0.914  (diff +0.004)
  seed 7: QNN 0.888 vs MLP 0.885  (diff +0.002)
  seed 123: QNN 0.908 vs MLP 0.941  (diff -0.033)
  seed 2024: QNN 0.935 vs MLP 0.948  (diff -0.014)
  seed 99: QNN 0.901 vs MLP 0.899  (diff +0.003)
  seed 5: QNN 0.888 vs MLP 0.898  (diff -0.010)
  seed 17: QNN 0.901 vs MLP 0.913  (diff -0.012)
  seed 31: QNN 0.881 vs MLP 0.893  (diff -0.012)
  seed 64: QNN 0.902 vs MLP 0.914  (diff -0.012)
  seed 88: QNN 0.891 vs MLP 0.916  (diff -0.025)
  mean diff -0.0108, 95% hier-CI [-0.0188,-0.0032], p=0.004
saved /kaggle/working/results/fig11_binary_auc.png


In [19]:
# ===================== BASE-RATE / PPV (Table 10) + CLASSICAL CEILING (Table 9) =====================
if True:
    print("== Base-rate / PPV (TPR 0.95, FPR 0.10) ==")
    TPR,FPR=0.95,0.10; rows=[]
    for prior in [0.909,0.05,0.01]:
        ppv=(TPR*prior)/(TPR*prior+FPR*(1-prior)); rows.append([prior,round(ppv,3)])
    savetab(pd.DataFrame(rows,columns=["attack_prior","PPV"]),"table10_baserate.csv"); print(rows)

    print("== Classical controls (PCA-8) + full-feature ceiling ==")
    # PCA-8 controls (Table 8)
    Etr,Ete,ys,ybs,_,_=prep(42,N_QUBITS,TRAIN_SUB,"angle",N_QUBITS)
    lr=LogisticRegression(max_iter=500).fit(Etr,ys); mlp=MLPClassifier((64,),max_iter=400,random_state=42).fit(Etr,ys)
    t8=pd.DataFrame([["LogReg",accuracy_score(yte,lr.predict(Ete)),f1_score(yte,lr.predict(Ete),average="weighted")],
                     ["MLP-64",accuracy_score(yte,mlp.predict(Ete)),f1_score(yte,mlp.predict(Ete),average="weighted")]],
                    columns=["Model","Acc","F1w"]); savetab(t8,"table8_classical_pca8.csv"); print(t8.to_string(index=False))
    # full-feature ceiling (Table 9) — try xgboost, else sklearn GradientBoosting
    sc=StandardScaler().fit(Xtr_raw); XtrS,XteS=sc.transform(Xtr_raw),sc.transform(Xte_raw); rows=[]
    try:
        pipq("xgboost"); from xgboost import XGBClassifier
        clf=XGBClassifier(n_estimators=300,max_depth=6,tree_method="hist",verbosity=0).fit(XtrS,ytr)
        rows.append(["XGBoost(full)",accuracy_score(yte,clf.predict(XteS))])
    except Exception as e: print("  xgboost skipped:",e)
    from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
    hgb=HistGradientBoostingClassifier(max_iter=300).fit(XtrS,ytr); rows.append(["HistGBM(full)",accuracy_score(yte,hgb.predict(XteS))])
    savetab(pd.DataFrame(rows,columns=["Model","Acc"]),"table9_ceiling.csv"); print(rows)


== Base-rate / PPV (TPR 0.95, FPR 0.10) ==
saved /kaggle/working/results/table10_baserate.csv
[[0.909, 0.99], [0.05, 0.333], [0.01, 0.088]]
== Classical controls (PCA-8) + full-feature ceiling ==
saved /kaggle/working/results/table8_classical_pca8.csv
 Model      Acc      F1w
LogReg 0.540909 0.490794
MLP-64 0.579545 0.535128
saved /kaggle/working/results/table9_ceiling.csv
[['XGBoost(full)', 0.9854545454545455], ['HistGBM(full)', 0.9818181818181818]]


In [20]:
# ===================== package =====================
import shutil
base=os.path.join("/kaggle/working" if os.path.isdir("/kaggle/working") else ".","DNP3_WP_results")
shutil.make_archive(base,"zip",RESULTS)
print("\n==== DONE ====\nresults in:",RESULTS,"\ndownload & send back:",base+".zip")
for f in sorted(os.listdir(RESULTS)): print("  -",f)



==== DONE ====
results in: /kaggle/working/results 
download & send back: /kaggle/working/DNP3_WP_results.zip
  - binary_hier_bootstrap.json
  - data_audit.json
  - fig11_binary_auc.png
  - fig5_11class.png
  - fig6_amp_confusion.png
  - fig7_dru_confusion.png
  - fig8_5class_confusion.png
  - fig_pca_variance.png
  - pca_variance.csv
  - table10_baserate.csv
  - table11_binary_seed42.csv
  - table5_11class_seed42.csv
  - table6_11class_3seed.csv
  - table7_5class.json
  - table7_macro.json
  - table8_classical_pca8.csv
  - table9_ceiling.csv
  - wp1_matched8.json
  - wp1_matched8.npz
  - wp2_encoding_control.png
  - wp2_matched_sweep.csv
  - wp3_dedup_splits.csv
  - wp3_summary.json
  - wp4_finite_shots.csv
  - wp4_gate_counts.csv
  - wp4_scaling.csv
  - wp4_scaling.png
  - wp5_noise.csv
  - wp6_drift.csv
  - wp6_fgsm.csv
  - wp6_fgsm.png
